In [ ]:
import os
import pandas as pd
from tqdm.notebook import tqdm

from utilities import get_date, walk_through_files, get_subzone_name_from_dir

pd.set_option("display.max_rows", 500)

RECORDS_PATH = os.path.join("raw-data", "sg_subzones_devices")
SAVEPATH = os.path.join("intermediate-files", "devicecount-subzone-day.csv")

In [ ]:
ix = 0
dict_placeholder = {}


for filepath in tqdm(walk_through_files(RECORDS_PATH, "csv.gz", fullpath=True)):

    # parse file name to get date and loc
    try:
        subzone = get_subzone_name_from_dir(filepath)
        date = get_date(filepath)
    except ValueError:
        print(filepath)

    # read into df
    _df = pd.read_csv(
        filepath,
        compression="gzip",
        names=["id", "os"],
        dtype={"id": "object", "os": "int8"},
    )

    android_count = len(_df[_df["os"] == 0])
    ios_count = len(_df[_df["os"] == 1])

    # init list and add to dict
    dict_placeholder[ix] = [subzone, date, android_count, ios_count]

    ix += 1

In [ ]:
df = (
    pd.DataFrame.from_dict(
        dict_placeholder,
        orient="index",
        columns=["subzone", "date", "android_count", "ios_count"],
    )
    .astype(
        {
            "subzone": "object",
            "date": "datetime64[ns]",
            "android_count": "int",
            "ios_count": "int",
        }
    )
    .sort_values(["date", "subzone"], ignore_index=True)
)

In [ ]:
df.head(10)

In [ ]:
df[df["android_count"] < df["ios_count"]]

In [ ]:
df.groupby("subzone")["android_count"].sum().reset_index().sort_values(
    "android_count"
).head(15)

In [ ]:
df.groupby("subzone")["android_count"].sum().reset_index().sort_values(
    "android_count"
).tail(15)

In [ ]:
df.to_csv(SAVEPATH, index=False)